# Path B — Direct Redshift Pull (`psycopg2`)

> This is the **alternative** data-access path. The main module path is **Path A** in `eda_example.ipynb`, which uses `data_io.load_data()` (boto3 `redshift-data` + `UNLOAD` to S3, IAM auth, no VPN). See its Setup section for the comparison.

**Use this notebook when**: you need a quick one-off sample on your laptop and you already have VPN + DB credentials. The output Parquet is what gets committed to `sample_data_from_redshift/` for offline development.

## Steps

1. **Connect to VPN** — `vpn.ao.zapsi.net`.
2. **Verify the connection** in the AWS Redshift Query Editor:
   <https://af-south-1.console.aws.amazon.com/sqlworkbench/home?region=af-south-1#/client>
3. **Provide the password via environment variable** — never hardcode it in the notebook.
   Create a `.env` file in this folder (already in `.gitignore`):
   ```
   REDSHIFT_PASSWORD=...
   ```
   Or export it in your shell before launching Jupyter:
   ```bash
   export REDSHIFT_PASSWORD='...'
   ```
4. **Install dependencies** (if not already installed via `requirements.txt`):
   ```bash
   pip install python-dotenv psycopg2-binary pyarrow
   ```
   Restart the kernel after installing.
5. **Run the cell below** — it connects, runs the sample query, and saves the result to `../sample_data_from_redshift/`.

⚠️ **Security**: the password is a real production credential. Read it from the environment, never paste it into the notebook. If it ever lands in a committed cell, **rotate it immediately** — git history retains it forever.

In [ ]:
from dotenv import load_dotenv
import os
import psycopg2
import pandas as pd

# Loads the .env at the repo root (or any parent), pulling in `redshift_password`
load_dotenv()

# ====== CONFIG ======
REDSHIFT_HOST = "redshift-cluster-dsi.cl4o4mmtx9ir.af-south-1.redshift.amazonaws.com"
REDSHIFT_PORT = 5439
REDSHIFT_DB   = "prod"
REDSHIFT_USER = "awsuser"

# Read the password from the .env file — never hardcode it here.
REDSHIFT_PASSWORD = os.getenv("redshift_password")

if not REDSHIFT_PASSWORD:
    raise RuntimeError(
        "redshift_password is not set. Create a .env file at the repo root "
        "containing:\n    redshift_password=\"your-password\""
    )

# Example query — change to your table
QUERY = "SELECT * FROM prod.dth_churn_ml_training.training_features WHERE RANDOM() < 0.003;"

OUTPUT_PARQUET = "../sample_data_from_redshift/sample_from_prod.parquet"
# ====================

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD,
)

try:
    df = pd.read_sql(QUERY, conn)
    print(f"Fetched {len(df)} rows from Redshift")
    print(df.head())

    df.to_parquet(OUTPUT_PARQUET, index=False)
    print(f"Saved {len(df)} rows to {OUTPUT_PARQUET}")
finally:
    conn.close()